# Module 6: Policy Gradients and Actor-Critic

**Spinning Up in Active Inference** | Part 2: The RL Track

---

## Historical Context

Everything we've seen so far—Q-learning, SARSA, Dyna—learns a **value function** and derives a policy from it. But in 1992, Ronald Williams proposed a radical alternative: what if we parameterize the policy directly and optimize it with gradient ascent?

The **REINFORCE** algorithm (Williams, 1992) was elegant but suffered from high variance. The solution came from an idea with deep roots in control theory: use a **critic** (value function) to reduce variance while the **actor** (policy) does the actual decision-making. This **Actor-Critic** architecture, refined by Sutton, McAllester, Singh & Mansour (2000), became the foundation of modern deep RL.

The connection to Active Inference is striking: AIF's policy selection $\pi(a) \propto \exp(-G(a))$ is a softmax policy—exactly an energy-based policy gradient method. And AIF's information-seeking behavior emerges from what RL calls **entropy regularization**.

### What You'll Learn

1. The shift from value-based to policy-based methods
2. The REINFORCE algorithm and the policy gradient theorem
3. Variance reduction through baselines and advantage functions
4. The Actor-Critic architecture
5. Entropy regularization and its deep connection to free energy minimization

## 6.1 Why Policy Gradients?

Value-based methods (TDQ, SARSA) have a fundamental limitation: they compute $Q(s,a)$ for all actions and then derive a policy via $\arg\max$ or softmax. This is problematic when:

- **Continuous action spaces**: You can't enumerate all actions
- **Stochastic policies are needed**: Sometimes randomness IS the optimal strategy (rock-paper-scissors)
- **The policy has simpler structure than the value function**: A policy might be "go right" everywhere, even though $V(s)$ is complicated

Policy gradient methods parameterize the policy directly:

$$\pi_\theta(a|s) = \text{some differentiable function of } \theta$$

and optimize by gradient ascent on expected return:

$$\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)$$

where $J(\theta) = \mathbb{E}_{\pi_\theta}\left[\sum_t \gamma^t r_t\right]$.

## 6.2 The Policy Gradient Theorem

The remarkable **policy gradient theorem** (Sutton et al., 2000) tells us:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[\nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

where $G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$ is the return from time $t$.

**Intuition**: Increase the probability of actions that led to high returns. Decrease the probability of actions that led to low returns. The $\nabla \log \pi$ term points in the direction that increases the action's probability; $G_t$ scales how much.

### REINFORCE (Monte Carlo Policy Gradient)

```
for each episode:
    Generate trajectory: s_0, a_0, r_0, s_1, a_1, r_1, ...
    for each step t:
        G_t = sum of discounted future rewards
        theta += alpha * grad(log pi(a_t|s_t)) * G_t
```

Simple, elegant—but high variance because $G_t$ is a noisy estimate.

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from utils.plotting import (plot_learning_curve, plot_policy_distribution,
                            plot_value_heatmap, dual_perspective_box, COLORS)

np.random.seed(42)
print("Module 6: Policy Gradients and Actor-Critic")
print("="*50)

## 6.3 REINFORCE from Scratch: A Multi-Armed Bandit

Let's implement REINFORCE on the simplest possible problem: a **multi-armed bandit** with 4 arms. There are no states, just actions—so the policy is simply a probability distribution over arms.

We parameterize: $\pi_\theta(a) = \text{softmax}(\theta)$

In [ ]:
def softmax(x):
    """Numerically stable softmax."""
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()


class REINFORCEBandit:
    """REINFORCE on a stateless bandit problem."""

    def __init__(self, n_actions, lr=0.1):
        self.n_actions = n_actions
        self.lr = lr
        self.theta = np.zeros(n_actions)  # Policy parameters
        self.policy_history = []  # Track policy evolution

    def get_policy(self):
        return softmax(self.theta)

    def sample_action(self):
        pi = self.get_policy()
        return np.random.choice(self.n_actions, p=pi)

    def update(self, action, reward):
        pi = self.get_policy()
        # Policy gradient: grad log pi(a) * reward
        # For softmax: grad log pi(a) = e_a - pi
        #   where e_a is one-hot vector for action a
        grad_log_pi = np.zeros(self.n_actions)
        grad_log_pi[action] = 1.0  # e_a
        grad_log_pi -= pi           # - pi

        self.theta += self.lr * grad_log_pi * reward
        self.policy_history.append(self.get_policy().copy())


# Define a bandit: arm rewards (mean)
true_rewards = np.array([0.2, 0.5, 0.8, 0.3])  # Arm 2 is best
n_actions = len(true_rewards)

# Train REINFORCE
agent = REINFORCEBandit(n_actions, lr=0.05)
rewards_history = []

for step in range(500):
    action = agent.sample_action()
    reward = true_rewards[action] + np.random.randn() * 0.1  # Noisy reward
    agent.update(action, reward)
    rewards_history.append(reward)

print(f"True rewards:  {true_rewards}")
print(f"Final policy:  {agent.get_policy().round(3)}")
print(f"Best arm: {np.argmax(true_rewards)} | Agent's choice: {np.argmax(agent.get_policy())}")

In [ ]:
# Visualize policy evolution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Policy over time
policy_arr = np.array(agent.policy_history)
arm_labels = [f"Arm {i}\n(r={true_rewards[i]})" for i in range(n_actions)]
colors_arms = ['#E57373', '#4FC3F7', '#81C784', '#FFB74D']
for i in range(n_actions):
    axes[0].plot(policy_arr[:, i], label=arm_labels[i], linewidth=2, color=colors_arms[i])
axes[0].set_xlabel("Step")
axes[0].set_ylabel("P(action)")
axes[0].set_title("REINFORCE: Policy Evolution")
axes[0].legend(fontsize=8)
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(True, alpha=0.3)

# Final policy
plot_policy_distribution(agent.get_policy(), action_names=arm_labels,
                         title="Final Policy", ax=axes[1])

# Smoothed reward over time
window = 20
smoothed = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
axes[2].plot(smoothed, color=COLORS['reward'], linewidth=2)
axes[2].axhline(y=max(true_rewards), color='red', linestyle='--',
                alpha=0.5, label=f"Optimal ({max(true_rewards)})")
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Reward (smoothed)")
axes[2].set_title("Average Reward Over Time")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("REINFORCE on a 4-Armed Bandit", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6.4 Variance Reduction: Baselines and Advantages

REINFORCE works but is **high variance**: the gradient estimate $\nabla \log \pi(a|s) \cdot G_t$ fluctuates wildly because $G_t$ can vary enormously across episodes.

### Baseline Subtraction

A key insight: subtracting a **baseline** $b(s)$ from $G_t$ doesn't change the expected gradient but reduces variance:

$$\nabla_\theta J = \mathbb{E}\left[\nabla_\theta \log \pi_\theta(a_t|s_t) \cdot (G_t - b(s_t))\right]$$

The optimal baseline is approximately $b(s) \approx V(s)$. The term $G_t - V(s_t)$ is called the **advantage**:

$$A(s,a) = Q(s,a) - V(s)$$

It tells us: "How much better was this action compared to the average action in this state?" This is the foundation of the **Actor-Critic** architecture.

## 6.5 Actor-Critic on FourRooms

The **Actor-Critic** architecture has two components:

- **Actor** $\pi_\theta(a|s)$: the policy, updated via policy gradients
- **Critic** $V_w(s)$: the value function, updated via TD learning

The critic provides the baseline for variance reduction:

$$\delta_t = r_t + \gamma V_w(s_{t+1}) - V_w(s_t) \quad \text{(TD error)}$$
$$\theta \leftarrow \theta + \alpha_{\text{actor}} \cdot \nabla \log \pi_\theta(a_t|s_t) \cdot \delta_t$$
$$w \leftarrow w + \alpha_{\text{critic}} \cdot \delta_t \cdot \nabla V_w(s_t)$$

Let's use neuro-nav's `TDAC` (TD Actor-Critic) agent.

In [ ]:
from neuronav.agents.td_agents import TDQ, TDAC
from neuronav.envs.grid_env import GridEnv, GridSize, GridObservation
from neuronav.envs.grid_templates import GridTemplate

# Create environment
env = GridEnv(template=GridTemplate.four_rooms,
              obs_type=GridObservation.index,
              size=GridSize.small)
state_size = env.observation_space.n
action_size = env.action_space.n


def train_agent(agent_class, env, n_episodes=300, max_steps=500, **kwargs):
    """Train an agent and track per-episode rewards."""
    agent = agent_class(state_size, action_size, **kwargs)
    rewards = []
    for ep in range(n_episodes):
        obs = env.reset()
        total_r = 0
        for step in range(max_steps):
            action = agent.sample_action(obs)
            next_obs, reward, done, info = env.step(action)
            agent.update((obs, action, next_obs, reward, done))
            total_r += reward
            obs = next_obs
            if done:
                break
        rewards.append(total_r)
    return agent, rewards


# Train both agents
np.random.seed(42)
print("Training TDAC (Actor-Critic)...")
tdac_agent, tdac_rewards = train_agent(
    TDAC, env, n_episodes=300, lr=0.1, gamma=0.99, poltype="softmax", beta=5.0
)

np.random.seed(42)
print("Training TDQ (Q-learning)...")
tdq_agent, tdq_rewards = train_agent(
    TDQ, env, n_episodes=300, lr=0.1, gamma=0.99, poltype="softmax", beta=5.0
)

print(f"\nTDAC mean (last 50): {np.mean(tdac_rewards[-50:]):.3f}")
print(f"TDQ  mean (last 50): {np.mean(tdq_rewards[-50:]):.3f}")

In [ ]:
# Compare learning curves: Actor-Critic vs Q-learning
plot_learning_curve(
    {"TDAC (Actor-Critic)": tdac_rewards,
     "TDQ (Q-learning)": tdq_rewards},
    xlabel="Episode",
    ylabel="Cumulative Reward",
    title="Actor-Critic vs Q-Learning on FourRooms",
    window=20
)
plt.tight_layout()
plt.show()

print("Actor-Critic learns a policy directly; Q-learning learns values and derives a policy.")
print("Both converge, but their learning dynamics differ.")

## 6.6 Entropy of the Policy: Exploration to Exploitation

A key diagnostic for policy gradient methods is the **entropy** of the policy:

$$H(\pi(\cdot|s)) = -\sum_a \pi(a|s) \log \pi(a|s)$$

- **High entropy** = uniform policy = maximum exploration
- **Low entropy** = peaked policy = exploitation (confident)
- **Zero entropy** = deterministic policy

A healthy learning process typically shows entropy starting high (random exploration) and gradually decreasing as the agent discovers good actions.

This is deeply connected to Active Inference: the **precision** parameter $\gamma$ in AIF controls the sharpness of the policy $\pi(a) \propto \exp(-\gamma G(a))$. High precision = low entropy = exploitation.

In [ ]:
# Track policy entropy during training
def compute_policy_entropy(agent, state_size, action_size):
    """Compute average entropy of the agent's policy across all states."""
    entropies = []
    for s in range(state_size):
        # Get Q-values or policy for this state
        if hasattr(agent, 'q'):
            q_s = agent.q[s]
            # Softmax policy
            pi = softmax(q_s * 5.0)  # beta=5.0
        else:
            pi = np.ones(action_size) / action_size
        # Entropy
        pi_safe = np.clip(pi, 1e-10, None)
        h = -np.sum(pi_safe * np.log(pi_safe))
        entropies.append(h)
    return np.mean(entropies)


# Re-train TDAC while tracking entropy at intervals
np.random.seed(42)
env = GridEnv(template=GridTemplate.four_rooms,
              obs_type=GridObservation.index,
              size=GridSize.small)
state_size = env.observation_space.n
action_size = env.action_space.n

agent_entropy = TDAC(state_size, action_size, lr=0.1, gamma=0.99,
                     poltype="softmax", beta=5.0)
entropy_history = []
reward_history_ent = []
check_interval = 5

for ep in range(300):
    obs = env.reset()
    total_r = 0
    for step in range(500):
        action = agent_entropy.sample_action(obs)
        next_obs, reward, done, info = env.step(action)
        agent_entropy.update((obs, action, next_obs, reward, done))
        total_r += reward
        obs = next_obs
        if done:
            break
    reward_history_ent.append(total_r)
    if ep % check_interval == 0:
        entropy_history.append(compute_policy_entropy(
            agent_entropy, state_size, action_size))

# Plot entropy evolution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Entropy
max_entropy = np.log(action_size)
axes[0].plot(range(0, 300, check_interval), entropy_history,
             color=COLORS['epistemic'], linewidth=2, label='Policy Entropy')
axes[0].axhline(y=max_entropy, color='gray', linestyle='--', alpha=0.5,
                label=f'Max Entropy (uniform) = {max_entropy:.2f}')
axes[0].axhline(y=0, color='gray', linestyle=':', alpha=0.5,
                label='Min Entropy (deterministic) = 0')
axes[0].fill_between(range(0, 300, check_interval), entropy_history,
                     alpha=0.1, color=COLORS['epistemic'])
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("H(pi)")
axes[0].set_title("Policy Entropy Over Training")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Reward
smoothed_r = np.convolve(reward_history_ent, np.ones(20)/20, mode='valid')
axes[1].plot(smoothed_r, color=COLORS['reward'], linewidth=2)
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Reward (smoothed)")
axes[1].set_title("Reward Over Training")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Entropy Decreases as the Agent Learns", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nInitial entropy: {entropy_history[0]:.3f} (near uniform: {max_entropy:.3f})")
print(f"Final entropy:   {entropy_history[-1]:.3f} (more peaked, confident policy)")

## 6.7 Entropy Regularization: Don't Collapse Too Fast

A policy that collapses to a deterministic action too early will miss better alternatives. **Entropy regularization** adds an entropy bonus to the objective:

$$J_{\text{reg}} = J(\theta) + \beta H(\pi_\theta)$$

where $\beta > 0$ controls the strength of regularization.

### The Deep Connection to Free Energy

This is where policy gradients and Active Inference converge. The entropy-regularized RL objective:

$$J_{\text{reg}} = \mathbb{E}_{\pi}\left[\sum_t r_t\right] + \beta H(\pi)$$

is mathematically equivalent to minimizing the **KL divergence** between the policy and a prior:

$$J_{\text{reg}} \propto -\text{KL}\left[\pi_\theta(a|s) \| \frac{\exp(Q(s,a)/\beta)}{Z(s)}\right]$$

And in Active Inference, policy selection is:

$$\pi(a) \propto \exp(-\gamma G(a)) \cdot E(a)$$

where $E(a)$ is the prior over policies (habits). This is **the same** as entropy-regularized policy optimization with the free energy $G(a)$ replacing the Q-value.

In [ ]:
# Demonstrate the effect of entropy regularization
# We'll implement a simple softmax policy gradient with varying entropy coefficients

class EntRegPolicyGradient:
    """Policy gradient with entropy regularization on a grid task."""

    def __init__(self, n_states, n_actions, lr=0.05, gamma=0.99, entropy_coef=0.0):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = lr
        self.gamma = gamma
        self.entropy_coef = entropy_coef
        # Policy parameters: theta[s, a]
        self.theta = np.zeros((n_states, n_actions))
        # Value function (baseline)
        self.V = np.zeros(n_states)

    def get_policy(self, state):
        return softmax(self.theta[state])

    def sample_action(self, state):
        pi = self.get_policy(state)
        return np.random.choice(self.n_actions, p=pi)

    def update(self, trajectory):
        """Update from a complete episode trajectory."""
        states, actions, rewards = trajectory
        T = len(rewards)

        # Compute returns
        returns = np.zeros(T)
        G = 0
        for t in reversed(range(T)):
            G = rewards[t] + self.gamma * G
            returns[t] = G

        # Update policy and baseline
        for t in range(T):
            s, a = states[t], actions[t]
            pi = self.get_policy(s)
            advantage = returns[t] - self.V[s]

            # Policy gradient with entropy bonus
            grad_log_pi = np.zeros(self.n_actions)
            grad_log_pi[a] = 1.0
            grad_log_pi -= pi

            # Entropy gradient: grad H = -grad sum_a pi(a) log pi(a)
            # = -(log pi + 1) * grad pi = -(log pi + 1) * pi * (e_a - pi)
            # Simplified: entropy bonus pushes theta toward uniform
            entropy_grad = -pi * (np.log(pi + 1e-10) + 1)
            entropy_grad -= entropy_grad.mean()  # Center

            self.theta[s] += self.lr * (grad_log_pi * advantage +
                                         self.entropy_coef * entropy_grad)

            # Update baseline
            self.V[s] += 0.1 * (returns[t] - self.V[s])

    def avg_entropy(self):
        entropies = []
        for s in range(self.n_states):
            pi = self.get_policy(s)
            h = -np.sum(pi * np.log(pi + 1e-10))
            entropies.append(h)
        return np.mean(entropies)


# Compare different entropy coefficients
entropy_coefs = [0.0, 0.01, 0.1, 0.5]
results = {}
entropy_traces = {}

for beta in entropy_coefs:
    np.random.seed(42)
    env = GridEnv(template=GridTemplate.four_rooms,
                  obs_type=GridObservation.index,
                  size=GridSize.small)
    state_size = env.observation_space.n
    action_size = env.action_space.n

    agent = EntRegPolicyGradient(state_size, action_size,
                                 lr=0.05, gamma=0.99, entropy_coef=beta)
    rewards = []
    entropies = []

    for ep in range(200):
        obs = env.reset()
        states, actions, rews = [], [], []
        for step in range(300):
            action = agent.sample_action(obs)
            next_obs, reward, done, info = env.step(action)
            states.append(obs)
            actions.append(action)
            rews.append(reward)
            obs = next_obs
            if done:
                break
        agent.update((states, actions, rews))
        rewards.append(sum(rews))
        if ep % 5 == 0:
            entropies.append(agent.avg_entropy())

    label = f"beta={beta}"
    results[label] = rewards
    entropy_traces[label] = entropies

print("Training complete for all entropy coefficients.")

In [ ]:
# Plot entropy coefficient comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_beta = ['#E57373', '#4FC3F7', '#81C784', '#FFB74D']

# Learning curves
for i, (label, rewards) in enumerate(results.items()):
    smoothed = np.convolve(rewards, np.ones(15)/15, mode='valid')
    axes[0].plot(smoothed, label=label, linewidth=2, color=colors_beta[i])
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Reward (smoothed)")
axes[0].set_title("Reward: Effect of Entropy Regularization")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Entropy traces
for i, (label, ents) in enumerate(entropy_traces.items()):
    axes[1].plot(range(0, 200, 5), ents, label=label, linewidth=2, color=colors_beta[i])
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Average Policy Entropy")
axes[1].set_title("Policy Entropy Over Training")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Entropy Regularization: Exploration vs Exploitation",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("beta=0.0  : No entropy bonus. Policy collapses fast (may miss better options).")
print("beta=0.01 : Light regularization. Gentle exploration pressure.")
print("beta=0.1  : Moderate. Good balance of exploration and exploitation.")
print("beta=0.5  : Heavy. Too much exploration — hard to commit to good actions.")

## 6.8 The Key Equation: Softmax Policies in RL and AIF

Here is the mathematical heart of the RL–AIF connection for policy selection:

### RL: Softmax Q-values

$$\pi_{\text{RL}}(a|s) = \frac{\exp(Q(s,a) / \tau)}{\sum_{a'} \exp(Q(s,a') / \tau)}$$

where $\tau$ is temperature (controls exploration).

### AIF: Softmax negative free energy

$$\pi_{\text{AIF}}(a) = \frac{\exp(-\gamma \cdot G(a)) \cdot E(a)}{\sum_{a'} \exp(-\gamma \cdot G(a')) \cdot E(a')}$$

where $G(a)$ is the expected free energy and $E(a)$ is the policy prior.

### The correspondence:

| RL | AIF |
|---|---|
| $Q(s,a)$ | $-G(a)$ = negative expected free energy |
| $\tau$ (temperature) | $1/\gamma$ (inverse precision) |
| Uniform prior over actions | $E(a)$ (policy prior / habits) |
| Entropy regularization $\beta H(\pi)$ | KL divergence from policy prior $\text{KL}[\pi \| E]$ |

When $E(a) = \text{uniform}$ and we set $Q(s,a) = -G(a)$, these are **identical**.

In [ ]:
# Demonstrate the equivalence: softmax(Q/tau) == softmax(-gamma*G) * E
from alf.free_energy import expected_free_energy_decomposed

# Simple 4-state, 4-action environment
n_s, n_a = 4, 4

# Observation model (slightly noisy identity)
A = np.array([
    [0.85, 0.05, 0.05, 0.05],
    [0.05, 0.85, 0.05, 0.05],
    [0.05, 0.05, 0.85, 0.05],
    [0.05, 0.05, 0.05, 0.85],
])

# Transition model: each action deterministically moves to that state
B = np.zeros((n_s, n_s, n_a))
for a in range(n_a):
    B[a, :, a] = 1.0  # Action a -> state a

# Preferences: prefer state/obs 2
C = np.log(np.array([0.05, 0.05, 0.85, 0.05]) + 1e-16)

# Current belief: in state 0
beliefs = np.array([0.9, 0.05, 0.025, 0.025])

# Compute EFE for each action
G_values = []
for a in range(n_a):
    decomp = expected_free_energy_decomposed(A, B, C, beliefs, a)
    G_values.append(decomp.G_total)
G_values = np.array(G_values)

# AIF policy: softmax(-gamma * G)
gamma_aif = 4.0
pi_aif = softmax(-gamma_aif * G_values)

# RL equivalent: Q = -G, tau = 1/gamma
Q_equiv = -G_values
tau_rl = 1.0 / gamma_aif
pi_rl = softmax(Q_equiv / tau_rl)

# Plot side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
action_labels = [f"a{i} (go to s{i})" for i in range(n_a)]

axes[0].bar(range(n_a), -G_values, color=COLORS['rl'], alpha=0.85)
axes[0].set_xticks(range(n_a))
axes[0].set_xticklabels(action_labels, rotation=20, ha='right', fontsize=9)
axes[0].set_title("RL: Q(s,a) = -G(a)")
axes[0].set_ylabel("Q-value")
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(n_a), pi_rl, color=COLORS['rl'], alpha=0.85, label='RL')
axes[1].bar(range(n_a), pi_aif, color=COLORS['aif'], alpha=0.45, label='AIF')
axes[1].set_xticks(range(n_a))
axes[1].set_xticklabels(action_labels, rotation=20, ha='right', fontsize=9)
axes[1].set_title(f"Policy: softmax(Q/tau) vs softmax(-gamma*G)")
axes[1].set_ylabel("P(action)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].bar(range(n_a), np.abs(pi_rl - pi_aif), color=COLORS['neutral'], alpha=0.85)
axes[2].set_xticks(range(n_a))
axes[2].set_xticklabels(action_labels, rotation=20, ha='right', fontsize=9)
axes[2].set_title("Difference: |pi_RL - pi_AIF|")
axes[2].set_ylabel("|Difference|")
axes[2].grid(True, alpha=0.3)

plt.suptitle("Softmax Policy: RL and AIF Are Identical",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Max |pi_RL - pi_AIF| = {np.max(np.abs(pi_rl - pi_aif)):.2e}")
print("They are the same policy! Q = -G, temperature = 1/gamma.")

In [ ]:
# Dual Perspective Box
dual_perspective_box(
    rl_text=(
        "The softmax policy <b>pi(a|s) = exp(Q/tau) / Z</b> with entropy regularization "
        "<b>J + beta*H(pi)</b> prevents premature convergence. The temperature tau controls "
        "the exploration-exploitation tradeoff. This is equivalent to minimizing "
        "KL[pi || exp(Q/tau)/Z], a free-energy-like objective."
    ),
    aif_text=(
        "The AIF policy <b>pi(a) = exp(-gamma*G(a)) * E(a) / Z</b> is mathematically "
        "identical to an energy-based policy. Entropy regularization in RL is equivalent "
        "to the KL divergence from the policy prior in free energy: "
        "<b>F = E[ln q(s) - ln p(o,s)]</b>. Both prevent the policy from collapsing."
    ),
    title="Entropy Regularization IS Free Energy Minimization"
)

## Exercises

### Exercise 6.1: REINFORCE with Baseline (Guided)
Modify the `REINFORCEBandit` class to subtract a running average reward as a baseline. Compare variance of the gradient estimates with and without the baseline by plotting the magnitude of $\theta$ updates over training.

### Exercise 6.2: Temperature Sweep (Intermediate)
Using the AIF EFE computation from cell 6.8, sweep the precision parameter $\gamma$ from 0.1 to 100. Plot how the policy entropy changes. At what precision does the policy become effectively deterministic (entropy < 0.1 nats)?

*Hint: This is the same as sweeping temperature $\tau = 1/\gamma$ in the RL softmax policy.*

### Exercise 6.3: Actor-Critic from Scratch (Open-ended)
Implement a tabular Actor-Critic agent without using neuro-nav. You need:
- Actor: $\theta[s, a]$ updated via policy gradient with TD error as advantage
- Critic: $V[s]$ updated via TD(0)

Compare to neuro-nav's TDAC on the same environment.

---

## Key Takeaways

1. **Policy gradient methods** optimize the policy directly via $\nabla J = \mathbb{E}[\nabla \log \pi \cdot G_t]$
2. **Baselines** (especially the value function) reduce variance without introducing bias
3. **Actor-Critic** combines policy gradients (actor) with TD learning (critic)
4. **Entropy regularization** $J + \beta H(\pi)$ in RL is equivalent to the KL term in free energy
5. The softmax policy $\pi(a) \propto \exp(-\gamma G(a))$ in AIF IS the entropy-regularized softmax policy in RL

## Further Reading

- Williams, R.J. (1992). Simple statistical gradient-following algorithms for connectionist reinforcement learning. *Machine Learning*, 8. — The REINFORCE algorithm.
- Sutton, R.S. et al. (2000). Policy gradient methods for RL with function approximation. *NeurIPS*. — The policy gradient theorem.
- Haarnoja, T. et al. (2018). Soft Actor-Critic: Off-policy maximum entropy deep RL. *ICML*. — Maximum entropy RL, deepening the connection to free energy.
- Millidge, B. (2020). Deep active inference as variational policy gradients. *JMLR*. — Direct proof that AIF policy selection = variational policy gradients.
- Friston, K.J. et al. (2012). Active inference and agency. *Cognitive Neuroscience*. — Precision as inverse temperature in Active Inference.

---

*Next: Module 7 — Deep RL and Representation Learning*